In [ ]:
# === Cell 1: Mount Drive, install nnU-Net v2, set env vars ===

from google.colab import drive
drive.mount('/content/drive')

!pip install nnunetv2

import os

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
os.makedirs(os.path.join(WORKSPACE, 'nnUNet_raw'), exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, 'nnUNet_preprocessed'), exist_ok=True)

for run_id in range(3):
    os.makedirs(os.path.join(WORKSPACE, f'nnUNet_results_run{run_id}'), exist_ok=True)

os.environ['nnUNet_raw'] = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
# nnUNet_results will be set per-run in Cell 6; default to run0 for now
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run0')

print('nnUNet_raw:', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed:', os.environ['nnUNet_preprocessed'])
print('nnUNet_results (default):', os.environ['nnUNet_results'])

In [ ]:
# === Cell 2: Convert dataset to nnU-Net format ===

import os, shutil, json
import numpy as np
from PIL import Image

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RAW = os.environ['nnUNet_raw']
DATASET = os.path.join(RAW, 'Dataset501_VFSS')

imagesTr = os.path.join(DATASET, 'imagesTr')
labelsTr = os.path.join(DATASET, 'labelsTr')
imagesTs = os.path.join(DATASET, 'imagesTs')

# Clear any existing contents to avoid stale files from previous runs
for d in [imagesTr, labelsTr, imagesTs]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Copy train + val images; remap masks to {0,1}
num_training = 0
for split in ['train', 'val']:
    img_dir = os.path.join(BASE, split, 'images')
    msk_dir = os.path.join(BASE, split, 'masks')
    for fname in sorted(os.listdir(img_dir)):
        if not fname.endswith('.png'):
            continue
        stem = fname.replace('.png', '')

        # Image: copy as-is with _0000 suffix
        shutil.copy2(
            os.path.join(img_dir, fname),
            os.path.join(imagesTr, f'{stem}_0000.png'),
        )

        # Mask: load, remap to {0,1}, save as uint8 PNG
        mask = np.array(Image.open(os.path.join(msk_dir, fname)).convert('L'))
        mask_bin = (mask > 0).astype(np.uint8)  # {0,255} -> {0,1}
        Image.fromarray(mask_bin).save(os.path.join(labelsTr, f'{stem}.png'))

        num_training += 1

print(f'Copied {num_training} training+val cases to imagesTr/labelsTr')

# Verify ALL converted labels contain only {0,1}
for label_fname in sorted(os.listdir(labelsTr)):
    label_arr = np.array(Image.open(os.path.join(labelsTr, label_fname)))
    unique_vals = set(np.unique(label_arr))
    assert unique_vals.issubset({0, 1}), (
        f'Label {label_fname} has unexpected values: {unique_vals}'
    )
print(f'Verified: all {num_training} labels contain only values in {{0, 1}}')

# Copy test images into imagesTs
num_test = 0
test_img_dir = os.path.join(BASE, 'test', 'images')
for fname in sorted(os.listdir(test_img_dir)):
    if not fname.endswith('.png'):
        continue
    stem = fname.replace('.png', '')
    shutil.copy2(
        os.path.join(test_img_dir, fname),
        os.path.join(imagesTs, f'{stem}_0000.png'),
    )
    num_test += 1

print(f'Copied {num_test} test cases to imagesTs')

# Create dataset.json
dataset_json = {
    'channel_names': {'0': 'Xray'},
    'labels': {'background': 0, 'vertebra': 1},
    'numTraining': num_training,
    'file_ending': '.png',
}

with open(os.path.join(DATASET, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'dataset.json saved with numTraining={num_training}')

In [ ]:
# === Cell 3: Build custom split (train/val from original split) ===

import os, json

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'

train_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'train', 'images'))
    if f.endswith('.png')
])

val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])

splits_final = [{
    'train': train_stems,
    'val': val_stems,
}]

print(f'Train stems: {len(train_stems)}')
print(f'Val stems:   {len(val_stems)}')
print(f'Total:       {len(train_stems) + len(val_stems)}')
print(f'First train: {train_stems[:3]}')
print(f'First val:   {val_stems[:3]}')

# Save temporarily; will copy to preprocessed folder after Cell 4
_SPLITS_TMP = '/content/splits_final.json'
with open(_SPLITS_TMP, 'w') as f:
    json.dump(splits_final, f, indent=2)
print(f'Temporary splits saved to {_SPLITS_TMP}')

In [ ]:
# === Cell 4: Planning and preprocessing ===

!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity -c 2d

In [ ]:
# === Cell 5: Write custom splits_final.json to preprocessed folder ===

import os, json, shutil

preprocessed_dir = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS'
)
assert os.path.isdir(preprocessed_dir), (
    f'Preprocessed folder not found: {preprocessed_dir}. Run Cell 4 first.'
)

splits_dst = os.path.join(preprocessed_dir, 'splits_final.json')
shutil.copy2('/content/splits_final.json', splits_dst)

# Verify
with open(splits_dst) as f:
    splits = json.load(f)

assert len(splits) == 1, f'Expected 1 fold, got {len(splits)}'
print(f'splits_final.json written to {splits_dst}')
print(f'  train: {len(splits[0]["train"])} cases')
print(f'  val:   {len(splits[0]["val"])} cases')

# Cross-check with dataset.json
ds_path = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'dataset.json')
with open(ds_path) as f:
    ds = json.load(f)
n_split = len(splits[0]['train']) + len(splits[0]['val'])
assert ds['numTraining'] == n_split, (
    f'Mismatch: numTraining={ds["numTraining"]} vs split total={n_split}'
)
print(f'Verified: numTraining ({ds["numTraining"]}) == train+val ({n_split})')

In [ ]:
# === Cell 6: Train 3 independent runs ===
# Each run trains for 1000 epochs. Total training time may be ~9-18 hours on Colab.
# nnU-Net has no seed parameter; we run 3 times to estimate run-to-run variability
# arising from non-deterministic data augmentation and weight initialization.
#
# To resume a specific run if Colab disconnects:
#   os.environ['nnUNet_results'] = WORKSPACE + '/nnUNet_results_runX'
#   !nnUNetv2_train 501 2d 0 --c

import os, gc, torch, time

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'

for run_id in range(3):
    print(f'\n{"="*60}')
    print(f'  TRAINING RUN {run_id}/2')
    print(f'{"="*60}\n')

    gc.collect()
    torch.cuda.empty_cache()

    results_dir = os.path.join(WORKSPACE, f'nnUNet_results_run{run_id}')
    os.makedirs(results_dir, exist_ok=True)
    os.environ['nnUNet_results'] = results_dir
    print(f'nnUNet_results -> {results_dir}')

    t0 = time.time()
    !nnUNetv2_train 501 2d 0
    elapsed = time.time() - t0

    fold_dir = os.path.join(
        results_dir,
        'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'
    )
    print(f'\nRun {run_id} done in {elapsed/3600:.1f}h')
    print(f'Checkpoints: {fold_dir}')
    if os.path.isdir(fold_dir):
        print('Contents:', os.listdir(fold_dir))

print('\nAll 3 runs complete.')

In [ ]:
# === Cell 7: Predict on test set from each of the 3 models ===

import os, shutil

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')

for run_id in range(3):
    print(f'\n--- Predicting from run {run_id} ---')

    os.environ['nnUNet_results'] = os.path.join(WORKSPACE, f'nnUNet_results_run{run_id}')
    pred_dir = os.path.join(RESULTS_BASE, f'predictions_run{run_id}')

    # Clear any existing predictions to avoid stale files
    if os.path.isdir(pred_dir):
        shutil.rmtree(pred_dir)
    os.makedirs(pred_dir)

    !nnUNetv2_predict \
        -i {imagesTs} \
        -o {pred_dir} \
        -d 501 -c 2d -f 0

    preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
    print(f'Run {run_id}: {len(preds)} predictions saved to {pred_dir}')

print('\nAll predictions complete.')

In [ ]:
# === Cell 8: Evaluate all 3 runs, compute mean +/- std, save results ===

import os, json, csv, re, shutil, time
import numpy as np
from PIL import Image

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
GT_DIR = os.path.join(BASE, 'test', 'masks')
os.makedirs(RESULTS_BASE, exist_ok=True)

gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    """Strip .png and any trailing _NNNN channel suffix (e.g. _0000)."""
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


all_run_results = []
all_run_per_image = []

for run_id in range(3):
    pred_dir = os.path.join(RESULTS_BASE, f'predictions_run{run_id}')
    pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
    pred_by_stem = {
        normalize_pred_stem(f): os.path.join(pred_dir, f)
        for f in pred_files
    }

    matched_stems = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
    assert len(matched_stems) == len(gt_files), (
        f'Run {run_id}: expected {len(gt_files)} matches, got {len(matched_stems)}. '
        f'Unmatched GT: {set(gt_by_stem.keys()) - set(pred_by_stem.keys())}'
    )

    per_image = []
    for stem in matched_stems:
        gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
        pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))

        if pred.shape != gt.shape:
            pred = np.array(
                Image.fromarray(pred).resize(
                    (gt.shape[1], gt.shape[0]), Image.NEAREST
                )
            )

        gt_bin = (gt > 0).astype(np.float32)
        pred_bin = (pred > 0).astype(np.float32)
        dice, iou = compute_dice_iou(pred_bin, gt_bin)
        per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

    mean_f1 = np.mean([r['f1'] for r in per_image])
    mean_iou = np.mean([r['iou'] for r in per_image])

    all_run_results.append({'run_id': run_id, 'mean_f1': mean_f1, 'mean_iou': mean_iou})
    all_run_per_image.append(per_image)

    # Save per-run CSV
    csv_path = os.path.join(RESULTS_BASE, f'per_image_metrics_run{run_id}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
        writer.writeheader()
        writer.writerows(per_image)

    print(f'Run {run_id}: F1={mean_f1:.4f}, IoU={mean_iou:.4f} ({len(per_image)} images)')

# Aggregate across 3 runs
f1_values = [r['mean_f1'] for r in all_run_results]
iou_values = [r['mean_iou'] for r in all_run_results]

agg_f1_mean = float(np.mean(f1_values))
agg_f1_std = float(np.std(f1_values))
agg_iou_mean = float(np.mean(iou_values))
agg_iou_std = float(np.std(iou_values))

print(f'\n{"="*50}')
print(f'Aggregated (3 runs):')
print(f'  F1 (Dice): {agg_f1_mean:.4f} +/- {agg_f1_std:.4f}')
print(f'  IoU:       {agg_iou_mean:.4f} +/- {agg_iou_std:.4f}')
print(f'{"="*50}')

# Save aggregated metrics.json
metrics = {
    'mean_f1': agg_f1_mean,
    'std_f1': agg_f1_std,
    'mean_iou': agg_iou_mean,
    'std_iou': agg_iou_std,
    'per_run': [
        {'run_id': r['run_id'], 'mean_f1': float(r['mean_f1']), 'mean_iou': float(r['mean_iou'])}
        for r in all_run_results
    ],
}
with open(os.path.join(RESULTS_BASE, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved: metrics.json')

# Load nnUNet plans for run_report
plans_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'nnUNetPlans.json'
)
plans_summary = {}
if os.path.isfile(plans_path):
    with open(plans_path) as f:
        plans = json.load(f)
    cfg_2d = plans.get('configurations', {}).get('2d', {})
    plans_summary = {
        'patch_size': cfg_2d.get('patch_size'),
        'batch_size': cfg_2d.get('batch_size'),
        'architecture': cfg_2d.get('architecture'),
        'spacing': cfg_2d.get('spacing'),
    }
    shutil.copy2(plans_path, os.path.join(RESULTS_BASE, 'nnUNetPlans.json'))
    print('Copied nnUNetPlans.json to results folder')

# Build run_report.json
run_report = {
    'experiment': 'nnunet_baseline',
    'dataset_id': 501,
    'configuration': '2d',
    'num_runs': 3,
    'epochs_per_run': 1000,
    'mean_f1': agg_f1_mean,
    'std_f1': agg_f1_std,
    'mean_iou': agg_iou_mean,
    'std_iou': agg_iou_std,
    'per_run_results': [
        {'run_id': r['run_id'], 'mean_f1': float(r['mean_f1']), 'mean_iou': float(r['mean_iou'])}
        for r in all_run_results
    ],
    'num_test_images': len(gt_files),
    'nnunet_config_summary': plans_summary,
    'note': '3 independent runs to estimate run-to-run variability. '
            'nnU-Net has no seed parameter; variability may arise from '
            'non-deterministic data augmentation and weight initialization.',
    'paths': {
        'predictions_run0': os.path.join(RESULTS_BASE, 'predictions_run0'),
        'predictions_run1': os.path.join(RESULTS_BASE, 'predictions_run1'),
        'predictions_run2': os.path.join(RESULTS_BASE, 'predictions_run2'),
        'checkpoints_run0': os.path.join(WORKSPACE, 'nnUNet_results_run0',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'checkpoints_run1': os.path.join(WORKSPACE, 'nnUNet_results_run1',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'checkpoints_run2': os.path.join(WORKSPACE, 'nnUNet_results_run2',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'nnUNetPlans': plans_path,
    },
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open(os.path.join(RESULTS_BASE, 'run_report.json'), 'w') as f:
    json.dump(run_report, f, indent=2)
print(f'Saved: run_report.json')

# --- Final verification ---
print('\n--- Verification ---')

splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
if os.path.isfile(splits_path):
    with open(splits_path) as f:
        splits = json.load(f)
    assert len(splits) == 1, f'splits_final.json has {len(splits)} folds, expected 1'
    print('OK: splits_final.json has 1 fold')

    ds_path = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'dataset.json')
    with open(ds_path) as f:
        ds = json.load(f)
    n_split = len(splits[0]['train']) + len(splits[0]['val'])
    assert ds['numTraining'] == n_split
    print(f'OK: numTraining ({ds["numTraining"]}) == train+val ({n_split})')

for run_id in range(3):
    assert len(all_run_per_image[run_id]) == len(gt_files), (
        f'Run {run_id}: expected {len(gt_files)} results, got {len(all_run_per_image[run_id])}'
    )
print(f'OK: All 3 runs matched {len(gt_files)} test predictions to GT')

print(f'\nDone. F1={agg_f1_mean:.4f} +/- {agg_f1_std:.4f}, IoU={agg_iou_mean:.4f} +/- {agg_iou_std:.4f}')